# University Dataset — Data Cleaning & Preparation process 


# Overview

This notebook walks through a full **data cleaning pipeline** for the University Student Dataset, which contains **3,000 records** and **40 columns** covering academic performance, demographics, financial status, engagement, and career outcomes.   

### Libraries Used
- **`Pandas`** — core data manipulation (DataFrames, groupby, apply, fillna, etc.)
- **`NumPy`** — numerical operations and NaN handling
- **`IPython.display`** — render tables as styled HTML inside the notebook

# Pipeline Steps:

# Load The DataSet

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
df = pd.read_csv("University dataset.csv")
display(df)

,Student_ID,Age,Gender,Nationality,Program,Major_Specialization,Year_of_Study,Enrollment_Type,GPA,High_School_GPA,...,Financial_Aid,Family_Income,Parental_Education_Level,Health_Insurance,Mental_Health_Support_Used,Academic_Probation,Dropout_Risk_Score,Graduation_Expected_Year,Employment_Offer,Starting_Salary_Offer
0,1,30,Female,Canada,Mathematics,Cybersecurity,5,FullTime,2.99,3.53,...,No,126717.0,PhD,Yes,No,No,1.000,2027,No,0.0
1,2,19,Male,Canada,AI,Finance,2,FullTime,3.37,3.43,...,No,215708.0,Master,No,No,No,0.565,2028,Yes,45143.0
2,3,19,Female,UK,Physics,Marketing,2,PartTime,3.34,3.07,...,No,68245.0,HighSchool,No,No,No,0.670,2026,No,0.0
3,4,23,Male,Australia,Economics,Cybersecurity,1,FullTime,2.55,4.00,...,No,110478.0,PhD,Yes,No,No,0.352,2026,Yes,34040.0
4,5,34,Male,UAE,Computer Science,Robotics,5,PartTime,2.94,3.22,...,Yes,292894.0,HighSchool,No,No,No,0.662,2029,Yes,36345.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2996,20,Female,China,Computer Science,Civil,2,FullTime,3.46,3.75,...,Yes,173825.0,PhD,Yes,Yes,No,0.901,2032,Yes,60667.0
2996,2997,24,Male,China,Mathematics,Robotics,4,FullTime,3.78,3.63,...,Yes,59072.0,Master,Yes,No,No,0.489,2029,Yes,82588.0
2997,2998,23,Male,France,Engineering,Civil,2,FullTime,2.45,3.09,...,No,NaN,HighSchool,No,No,No,0.862,2031,No,0.0
2998,2999,25,Male,UK,AI,Marketing,3,FullTime,3.64,3.40,...,Yes,25654.0,HighSchool,No,No,No,0.794,2027,Yes,79952.0


# Check Data cleansing 

Before any cleaning can begin, we need to fully understand what the data looks like: its structure, any issues with types, duplicate rows, and missing values.  
Each sub-step below addresses one dimension of data cleansing.
    

- ## Check DataType

In [2]:
print("\nInitial Data Types:\n", df.dtypes)


Initial Data Types:
 Student_ID                      int64
Age                             int64
Gender                         object
Nationality                    object
Program                        object
Major_Specialization           object
Year_of_Study                   int64
Enrollment_Type                object
GPA                           float64
High_School_GPA               float64
SAT_Score                     float64
Credits_Completed               int64
Credits_This_Semester           int64
Scholarship_Status             object
Scholarship_Amount            float64
Accommodation_Type             object
Distance_From_Campus_km       float64
Attendance_Rate               float64
Study_Hours_Per_Week          float64
Part_Time_Job                  object
Weekly_Work_Hours             float64
Internship_Completed           object
Internship_Count                int64
Clubs_Joined                    int64
Leadership_Role                object
Online_Courses_Taken        


   
**All the data types are correct.**

- ## Check duplicate rows  


In [3]:
print("\nNumber of Duplicate Rows:", df.duplicated().sum())


Number of Duplicate Rows: 0


**No duplicate rows**  
 No rows need to be removed.

- ## Check missing values  

In [4]:
df_raw = df.copy()
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing Count': missing_counts.values,
    'Missing %': missing_pct.values
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False).reset_index(drop=True)

display(missing_df)

,Column,Missing Count,Missing %
0,Starting_Salary_Offer,478,15.93
1,SAT_Score,343,11.43
2,Scholarship_Amount,310,10.33
3,Weekly_Work_Hours,279,9.30
4,Family_Income,262,8.73
5,High_School_GPA,225,7.50
6,Distance_From_Campus_km,194,6.47
7,Attendance_Rate,191,6.37
8,Study_Hours_Per_Week,191,6.37
9,English_Proficiency_Score,166,5.53


**There are missing values in 11 columns.**   
No column exceeds 16% missing — the standard threshold below which **imputation** is preferred over column deletion.   
All 11 columns are worth keeping

- ## Check Outliers  

In [5]:
numeric_cols = df_raw.select_dtypes(include='number').columns.tolist()

# --- IQR-based outlier capping ---
outlier_log = {}
for col in numeric_cols:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    lower = max(0, lower)
    n_outliers = ((df_raw[col] < lower) | (df_raw[col] > upper)).sum()
    if n_outliers > 0:
        df_raw[col] = df_raw[col].clip(lower=lower, upper=upper)
        outlier_log[col] = {'outliers': int(n_outliers), 'lower': round(lower, 2), 'upper': round(upper, 2)}
total_outliers_capped = sum(v['outliers'] for v in outlier_log.values())
outlier_df = pd.DataFrame(outlier_log).T
outlier_df.index.name = "Column"
outlier_df.reset_index(inplace=True)

display(outlier_df)

,Column,outliers,lower,upper
0,Study_Hours_Per_Week,4.0,0.0,71.0
1,Weekly_Work_Hours,7.0,0.0,44.5
2,Family_Income,17.0,0.0,447504.5


**There are outliers in 3 columns.**

- ## Check the profiling of numeric columns

Summary statistics give us a quantitative sense of each column's central tendency, spread, and range.   
This is essential for choosing the right imputation strategy (mean vs. median) and for spotting any columns where the range suggests data entry errors.

In [6]:
numeric_cols = df_raw.select_dtypes(include='number').columns.tolist()

num_summary = df_raw[numeric_cols].describe().T[['mean', 'std', 'min', '50%', 'max']].round(2)
num_summary.columns = ['Mean', 'Std Dev', 'Min', 'Median', 'Max']
num_summary.index.name = 'Numeric Column'
display(num_summary)

,Mean,Std Dev,Min,Median,Max
Numeric Column,,,,,
Student_ID,1500.50,866.17,1.00,1500.50,3000.0
Age,25.62,5.23,17.00,26.00,34.0
Year_of_Study,2.99,1.40,1.00,3.00,5.0
GPA,3.00,0.58,1.77,3.01,4.0
High_School_GPA,3.25,0.43,2.36,3.24,4.0
SAT_Score,1251.31,202.43,900.00,1245.00,1693.0
Credits_Completed,90.52,51.93,0.00,92.00,179.0
Credits_This_Semester,14.47,3.40,9.00,14.00,20.0
Scholarship_Amount,13393.94,16545.58,0.00,2453.50,49927.0


- ## Check the profiling of categorical columns

In [7]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
cat_summary = pd.DataFrame({
    'Column': categorical_cols,
    'Unique Values': [df[c].nunique() for c in categorical_cols],
    'Top Value': [df[c].value_counts().index[0] for c in categorical_cols],
    'Top Value %': [(df[c].value_counts().iloc[0] / len(df) * 100).round(1).astype(str) + '%' for c in categorical_cols],
    'All Values': [', '.join(df[c].unique().astype(str)) if df[c].nunique() <= 8 else '...' for c in categorical_cols]
})
cat_summary = cat_summary.set_index('Column')
display(cat_summary)

,Unique Values,Top Value,Top Value %,All Values
Column,,,,
Gender,2,Male,50.5%,"Female, Male"
Nationality,10,China,10.9%,...
Program,7,Economics,14.9%,"Mathematics, AI, Physics, Economics, Computer ..."
Major_Specialization,7,Civil,14.9%,"Cybersecurity, Finance, Marketing, Robotics, D..."
Enrollment_Type,2,PartTime,50.9%,"FullTime, PartTime"
Scholarship_Status,2,Yes,52.4%,"Yes, No"
Accommodation_Type,3,OnCampus,33.6%,"OffCampus, WithFamily, OnCampus"
Part_Time_Job,2,Yes,51.6%,"Yes, No"
Internship_Completed,2,Yes,50.2%,"Yes, No"


# Remove irrelevant data

Before cleaning, we remove columns that do not provide useful information for analysis.   
Keeping them only increases file size and wastes memory.

In [9]:
df_clean=df.copy()

In [94]:
#df_clean = df_clean.drop(columns=[])

# Handel The Data

This section addresses the two most common data quality issues found above: **missing values** and **outliers**.   
Each issue is handled carefully using a clear and logical method.

- ## Handel the missing values

**Numeric** columns → filled with **median** imputation   
**Categorical** columns → filled with **mode**.imputation

In [10]:
numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()

numeric_imputed = {}

for col in numeric_cols:
    n_missing = df_clean[col].isnull().sum()

    if n_missing > 0:
        if col == "Starting_Salary_Offer":

            median_salary = df_clean.loc[
                df_clean["Employment_Offer"] == "Yes",
                col
            ].median()
            
            df_clean.loc[
                (df_clean[col].isnull()) & 
                (df_clean["Employment_Offer"] == "No"),
                col
            ] = 0

            df_clean.loc[
                (df_clean[col].isnull()) & 
                (df_clean["Employment_Offer"] == "Yes"),
                col
            ] = median_salary

            numeric_imputed[col] = (n_missing, round(median_salary, 2))

        elif col == "Scholarship_Amount":

            median_scholarship = df_clean.loc[
                df_clean["Scholarship_Status"] == "Yes",
                col
            ].median()

            df_clean.loc[
                (df_clean[col].isnull()) &
                (df_clean["Scholarship_Status"] == "No"),
                col
            ] = 0

            df_clean.loc[
                (df_clean[col].isnull()) &
                (df_clean["Scholarship_Status"] == "Yes"),
                col
            ] = median_scholarship

            numeric_imputed[col] = (n_missing, round(median_scholarship, 2))

        else:
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
            numeric_imputed[col] = (n_missing, round(median_val, 2))

cat_imputed = {}

for col in categorical_cols:
    n_missing = df_clean[col].isnull().sum()

    if n_missing > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        cat_imputed[col] = (n_missing, mode_val)

remaining_nulls = df_clean.isnull().sum().sum()
print(f"Remaining null values in dataset: {remaining_nulls}")

Remaining null values in dataset: 0


In [11]:
print("Numeric Columns Imputation:")
for col, (n_missing, median_val) in numeric_imputed.items():
    print(f"{col} → Replaced {n_missing} missing values with median = {median_val}")

Numeric Columns Imputation:
High_School_GPA → Replaced 225 missing values with median = 3.24
SAT_Score → Replaced 343 missing values with median = 1245.0
Scholarship_Amount → Replaced 310 missing values with median = 25725.0
Distance_From_Campus_km → Replaced 194 missing values with median = 24.5
Attendance_Rate → Replaced 191 missing values with median = 80.6
Study_Hours_Per_Week → Replaced 191 missing values with median = 27.0
Weekly_Work_Hours → Replaced 279 missing values with median = 15.0
English_Proficiency_Score → Replaced 166 missing values with median = 90.0
Family_Income → Replaced 262 missing values with median = 162477.5
Starting_Salary_Offer → Replaced 478 missing values with median = 72898.5


- ## Handel The Outliers Value using Capping Only
  **Re-checking Outliers on the Imputed Dataset**

  The IQR detection is re-run on `df_clean` (post-imputation). The bounds change slightly from the baseline check because imputed median values fill   in gaps in the distribution.

  ```
  Formula:
  Lower Bound = max(0, Q1 − 1.5 × IQR)   ← floored at 0 (no negative hours/income)
  Upper Bound = Q3 + 1.5 × IQR
  ```


In [14]:
numeric_cols = df_clean.select_dtypes(include='number').columns.tolist()

# --- IQR-based outlier capping ---
outlier_log = {}
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    lower = max(0, lower)
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    if n_outliers > 0:
        df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
        outlier_log[col] = {'outliers': int(n_outliers), 'lower': round(lower, 2), 'upper': round(upper, 2)}
total_outliers_capped = sum(v['outliers'] for v in outlier_log.values())
outlier_df = pd.DataFrame(outlier_log).T
outlier_df.index.name = "Column"
outlier_df.reset_index(inplace=True)

display(outlier_df)

,Column,outliers,lower,upper
0,Study_Hours_Per_Week,5.0,0.0,67.00
1,Weekly_Work_Hours,10.0,0.0,40.50
2,Family_Income,19.0,0.0,412628.88


### handel the outliers

In [15]:
for col in numeric_cols:
    if col in outlier_log: 
        lower = outlier_log[col]['lower']
        upper = outlier_log[col]['upper']
        df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

total_outliers_capped = sum(v['outliers'] for v in outlier_log.values())
print(f"Total outliers capped: {total_outliers_capped}")

Total outliers capped: 34


In [ ]:
Use clip(lower, upper) to cap outliers without dropping any rows

# Rename the columns name

The original column names use `snake_case` with underscores.  
Renaming them to **Title Case with spaces**.

In [16]:
df_clean.columns = (
    df_clean.columns
    .str.replace("_", " ")
    .str.title()
    .str.replace("Gpa", "GPA")
    .str.replace("Sat", "SAT")
)

# Save the cleaned data in a CSV file

In [17]:
output_file = "cleaned_University_dataset.csv"
df_clean.to_csv(output_file, index=False, encoding="utf-8")
print(f"\n New CSV file created successfully: {output_file}")


 New CSV file created successfully: cleaned_University_dataset.csv
